# Table of Contents
1  Image概述
2  Image operator(mask, clip, select, addBands, export)
3  ImageCollection概述
4  ImageCollection operator(filter, select, set/get, calculation, toList)
5  Example 1. Calculate NDVI for Jiangxi and Extract NDWI for poyang Lake between 1982 to 2021;
6  Example 2. Download huge ImageCollection from google earth engine; -> Download Images
7  Example 3. Extract NDVI values of climate stations from 1982 to 2021; -> Spatial analysis
8  Example 4. Supervised Classification -> Images Classification
9  Example 5. Unsupervised Classification (clustering) 

In [ ]:
import ee, os
import geemap
# ee.Initialize()
Map = geemap.Map()
Map = geemap.Map(center=(39, 110), zoom=4)

Map.add_basemap('SATELLITE')
Map

In [ ]:
# -------------------------------------------------------
# 2  ImageCollection 去云
#  Calculate NDVI for Jiangxi
# -------------------------------------------------------
year = [2010, 2020]
# doy = '-05-01', '-09-30'

# start_date = str(year[0]) + doy[0]
# end_date = str(year[1]) + doy[1]

# dt_range = ee.Filter.and(ee.Filter.calendarRange(year[0],year[1]), ee.Filter.calendarRange(6,8,'month'))

# jx_bound = ee.FeatureCollection('users/yehuigeo/jx_bound')
country = ee.FeatureCollection('projects/iye2hui/assets/Country_cn')# 江西省县编码15636*
jx_bound = country.filter(ee.Filter.stringContains('gb', '15636'))
roi = jx_bound.geometry()

In [ ]:
Map.addLayer(jx_bound,{}, 'jx_country')

In [ ]:
def cloudMaskL457(image):
    qaMask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturationMask = image.select('QA_RADSAT').eq(0)
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermalBand = image.select('ST_B6').multiply(0.00341802).add(149.0)

    return image.addBands(opticalBands, None, True)\
                 .addBands(thermalBand, None, True)\
                 .updateMask(qaMask).updateMask(saturationMask)\
                 .copyProperties(image, ["system:time_start",'system:id'])

def maskS2clouds(image):   # This function was used to mask the clouds for sentinel-2
    qa = image.select('QA60')

  # Bits 10 and 11 are clouds and cirrus, respectively.
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11

  # Both flags should be set to zero, indicating clear conditions.
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(
             qa.bitwiseAnd(cirrusBitMask).eq(0))

  # Return the masked and scaled data, without the QA bands.
    return image.updateMask(mask).divide(10000).select("B.*").copyProperties(image, ["system:time_start",'system:id'])

def maskL8sr(image):
    qaMask = image.select('QA_PIXEL').bitwiseAnd(int('11111',2)).eq(0);
    saturationMask = image.select('QA_RADSAT').eq(0);

#   // Apply the scaling factors to the appropriate bands.
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2);
    thermalBands = image.select('ST_B6').multiply(0.00341802).add(149.0);
    
    return image.addBands(opticalBands, None, True)\
                 .addBands(thermalBands, None, True)\
                 .updateMask(qaMask)\
                 .updateMask(saturationMask)\
                 .copyProperties(image, ["system:time_start",'system:id'])

In [ ]:
# Landsat 3 MSS Collection 1 Tier 1 Raw Scenes
# Dataset Availability
# 1978-03-05 - 1983-03-31
# Landsat 5+7 scenes
# Dataset Availability
# 1984 - 2012
# get landsat 5
collection5 = ee.ImageCollection('LANDSAT/LT05/C02/T1_L2') \
    .filterBounds(jx_bound) \
    .filter(ee.Filter.lt('CLOUD_COVER',10))\
    .filter(ee.Filter.calendarRange(year[0],year[1],'year')) \
    .filter(ee.Filter.calendarRange(6,8,'month')) \
    .map(cloudMaskL457) \
    .select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'])
# print('Landsat5')
print(collection5.size().getInfo())
# Landsat 7 Collection 1 Tier 1 Raw Scenes
# Dataset Availability
# 1999 - 2021
# get landsat 7
# https://developers.google.com/earth-engine/datasets/catalog/landsat
# https://www.usgs.gov/landsat-missions/landsat-collection-2
collection7 = ee.ImageCollection('LANDSAT/LE07/C02/T1_L2') \
    .filterBounds(jx_bound) \
    .filter(ee.Filter.lt('CLOUD_COVER',10))\
    .filter(ee.Filter.calendarRange(year[0],year[1],'year')) \
    .filter(ee.Filter.calendarRange(6,8,'month')) \
    .map(cloudMaskL457) \
    .select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'])

print(collection7.size().getInfo())

In [ ]:
# Landsat 8 Collection 1 Tier 1 Raw Scenes
# Dataset Availability
# 2013 - 2021
# get landsat 8
collection8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')\
    .filterBounds(jx_bound) \
    .filter(ee.Filter.lt('CLOUD_COVER',20))\
    .filter(ee.Filter.calendarRange(year[0],year[1],'year')) \
    .filter(ee.Filter.calendarRange(6,8,'month')) \
    .map(maskL8sr).select(['.*?B2', '.*?B3', '.*?B4', '.*?B5', '.*?B6', '.*?B7', 'pixel_qa'],['B1', 'B2', 'B3', 'B4', 'B5', 'B7', 'QA_PIXEL']) 
print(collection8.size().getInfo())
# Sentinel-2 (S2) Multispectral Instrument (MSI) 
# Dataset Availability
# Level-1C orthorectified top-of-atmosphere reflectance -> 2015-06-23 - present
# Level-2A orthorectified atmospherically corrected surface reflectance.
# get sentinel-2A
sentinel2 = ee.ImageCollection("COPERNICUS/S2_SR")\
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',20))\
    .filter(ee.Filter.calendarRange(year[0],year[1],'year')) \
    .filter(ee.Filter.calendarRange(6,8,'month')) \
    .map(maskS2clouds).filterBounds(jx_bound)\
    .select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12'],['B1', 'B2', 'B3', 'B4', 'B5', 'B7']) 
print(sentinel2.size().getInfo())

-----------unuse filter by CLOUD_COVER------------
LANDSAT/LT05/C02/T1_L2/LT05_119040_20100711
LANDSAT/LE07/C02/T1_L2/LE07_119040_20170706

In [ ]:
col5 = collection5.first()
col7 = collection7.first()
print(col5.get('system:id').getInfo())
print(col7.get('system:id').getInfo())

<!-- vegetation fraction cover calculation -->
# vfc = (NDVI - NDVImin) / (NDVImax - NDVImin)
# NDVImin=NDVISoil为裸土或无植被区的NDVI值；NDVImax = NDVIVeg为纯植被区的NDVI值

In [ ]:
def ndvi_cal(img):
    ndvi = img.normalizedDifference(['SR_B4','SR_B3']).rename('NDVI')
    xmin = ndvi.reduceRegion(**{
          'reducer': ee.Reducer.min(),
          'geometry': roi,
          'scale': 30,
          'maxPixels': 1e13
        }).get('NDVI')
    xmax = ndvi.reduceRegion(**{
          'reducer': ee.Reducer.max(),
          'geometry': roi,
          'scale': 30,
          'maxPixels': 1e13
        }).get('NDVI')
    return img.addBands(ndvi).set({"xmin":xmin, "xmax":xmax})
    
def vfc(img):
    fv = img.expression('(ndvi - xmin) / (xmax - xmin)',{
        'ndvi': img.select("NDVI"),
        'xmin': ee.Number(img.get("xmin")), # ee.Image(xmin)
        'xmax': ee.Number(img.get("xmax"))
        }).rename('vfc')  
    return img.addBands(fv)
    
# xmin = NDVISoil为裸土或无植被区的NDVI值；xmax = NDVIVeg为纯植被区的NDVI值
#     fv = ndvi.subtract(xmin).divide(xmax.subtract(xmin)).rename('vfc')
# 全年序列的FVC时，NDVISoil取2 月末的裸土纯净像元的NDVI值，NDVIVeg取7 月中旬的全植被覆盖的NDVI值
# 张喜旺, 吴炳方. 基于中高分辨率遥感的植被覆盖度时相变换方法[J]. 生态学报, 2015, 35(4): 1155-1164

In [ ]:
collection7 = ee.ImageCollection('LANDSAT/LE07/C02/T1_L2') \
    .filterBounds(jx_bound) \
    .filter(ee.Filter.lt('CLOUD_COVER',10))\
    .filter(ee.Filter.calendarRange(year[0],year[1],'year')) \
    .map(cloudMaskL457) \
    .select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'])

print(collection7.size().getInfo())

In [ ]:
roi = jx_bound.geometry()
max_c7 = collection7.map(ndvi_cal).select('NDVI').max()
min_c7 = collection7.map(ndvi_cal).select('NDVI').min()

In [ ]:
print(type(max_c7))

In [ ]:
Map.addLayer(max_c7,{}, 'max_c7')
Map

In [ ]:
roi = jx_bound.geometry()
ndvi = collection5.first().normalizedDifference(['SR_B4','SR_B3']).rename('NDVI')
xmin = ee.Number(ndvi.reduceRegion(**{
      'reducer': ee.Reducer.min(),
      'geometry': roi,
      'scale': 30,
      'maxPixels': 1e13
    }).get('NDVI'))
xmax = ee.Number(ndvi.reduceRegion(**{
      'reducer': ee.Reducer.max(),
      'geometry': roi,
      'scale': 30,
      'maxPixels': 1e13
    }).get('NDVI'))

In [ ]:
print(xmin.getInfo())
print(xmax.getInfo())

In [ ]:
palette = ['FFFFFF', 'CE7E45', 'DF923D', 'F1B555', 'FCD163', '99B718',
               '74A901', '66A000', '529400', '3E8601', '207401', '056201',
               '004C00', '023B01', '012E01', '011D01', '011301']

In [ ]:
Map.addLayer(ndvi,{'min': 0, 'max': 1,'palette': palette}, 'ndvi')

In [ ]:
ndvi7 = collection7.first().normalizedDifference(['SR_B4','SR_B3']).rename('NDVI')
Map.addLayer(ndvi7,{'min': 0, 'max': 1,'palette': palette}, 'ndvi7')

In [ ]:
Map

In [ ]:
roi = jx_bound.geomoetry()

In [ ]:
col_ndvi = collection7.map(ndvi_cal)
col_vfc = col_ndvi.filter(ee.Filter.notNull(["xmin", "xmax"])).map(vfc)

In [ ]:
Map.addLayer(col_vfc.select('vfc').median(),{}, 'fv')

In [ ]:
# fv = ndvi.subtract(xmin).divide(xmax.subtract(xmin)).rename('vfc')
fv = ndvi1.subtract(xmin).divide(xmax.subtract(xmin)).rename('FV')
palette = ['FFFFFF', 'CE7E45', 'DF923D', 'F1B555', 'FCD163', '99B718',
               '74A901', '66A000', '529400', '3E8601', '207401', '056201',
               '004C00', '023B01', '012E01', '011D01', '011301']
Map.addLayer(fv.clip(roi),{min: 0, max: 1,'palette': palette}, 'fv')

In [ ]:
Map.addLayer(ndvi1.clip(roi),{min: 0, max: 1}, 'ndvi1')

In [ ]:
results = ndvi1.reduceRegions(**{
    'collection': roi,
    'reducer': ee.Reducer.histogram(2000, 0.001),
    'scale': 30
})

feature = results.first()
histogram = ee.Dictionary(feature.get("histogram"))
means = ee.List(histogram.get("bucketMeans"))
count = ee.Array(histogram.get("histogram"))

accum = count.accum(0).toList()
maxPixe = accum.get(-1).getInfo()
p5_pixe = maxPixe*0.05
p5_index = accum.filter(ee.Filter.lt("item", p5_pixe)).size()
p5 = means.get(p5_index)

p95_pixe = maxPixe*0.95
p95_index = accum.filter(ee.Filter.lt("item", p95_pixe)).size()
p95 = means.get(p95_index)

In [ ]:
p5.getInfo()

In [ ]:
p95.getInfo()

In [ ]:
percentile = ndvi1.reduceRegions(**{
    'collection': roi,
    'reducer': ee.Reducer.percentile([5, 95]),
    'scale': 30,
})

feature = percentile.first()
p5 = feature.get("p5")
p95 = feature.get("p95")

In [ ]:
p5.getInfo()

In [ ]:
p95.getInfo()